In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from scipy import stats
from statsmodels.stats.diagnostic import lilliefors
from sklearn.metrics import cohen_kappa_score

import warnings
warnings.filterwarnings('ignore')

In [2]:
path = r'/Users/lilinfeng/Documents/MyDocuments/FuwaiCentralChina/P-ProPulseZeeFree-GuoXing/'
file = 'DataAnalysis-new-new.xlsx'
save_path = path+'Result-20250210/'

data = pd.read_excel(path+'Data/'+file, sheet_name='Sheet1')
p_info = pd.read_excel(path+'Data/'+file, sheet_name='Sheet2')

name = ['伪影等级评分','图像质量评分','医师诊断信心等级']

In [3]:
def testtest(data_1,data_2):
    #正态性检验
    KS_1, p_lili_1 = lilliefors(data_1)
    KS_2, p_lili_2 = lilliefors(data_2)
    if p_lili_1 >0.05 and p_lili_2 >0.05:
        #方差齐性检验
        fVal, p_SD = stats.levene(data_1, data_2, center='mean')
        #独立样本t检验
        if p_SD > 0.05:
            statistic, p_final = stats.ttest_ind(data_1, data_2) 
        else:
            statistic, p_final = stats.ttest_ind(data_1, data_2, equal_var=False)
    else:
        #秩和检验
        statistic, p_final = stats.mannwhitneyu(data_1, data_2) 
    return statistic,p_final

# 患者基本信息

In [4]:
# 总体数据的基本信息
allpositive = data['患者编号'].values
n_patient = len(allpositive)/4
print('Number of stair-step artifacts = %.d'% (len(allpositive)/4))
positive = list(set(allpositive))
print('Number of patients with stair-step artifacts = %.d'% (len(positive)))
male, female = p_info['性别'].value_counts()
print('Info about Gender: male = %.d (%.1f), female = %.d (%.1f)'% 
      (male,male*100/(male+female),female,female*100/(male+female)))

for item in ['体重','身高（m)','BMI','心率平均值','造影剂总量','CTDI','SSDE','DLP','TS-SNR','TS-CNR','ZF-SNR','ZF-CNR']:
    buf = p_info[item].values
    # 正态性检验
    KS, p_lili = lilliefors(buf)
    if p_lili > 0.05:
        print('Info about '+item+': %.2f±%.2f'% (np.mean(buf),np.std(buf)))
    else:
        print('Info about '+item+': %.2f[%.2f, %.2f]'% 
              (np.median(buf),np.percentile(buf,[75,25])[0],np.percentile(buf,[75,25])[1]))

Number of stair-step artifacts = 112
Number of patients with stair-step artifacts = 46
Info about Gender: male = 52 (54.7), female = 43 (45.3)
Info about 体重: 70.81±11.68
Info about 身高（m): 1.66[1.72, 1.60]
Info about BMI: 25.57±3.09
Info about 心率平均值: 68.68±10.62
Info about 造影剂总量: 65.00[70.00, 65.00]
Info about CTDI: 12.30[19.80, 7.76]
Info about SSDE: 16.80[27.30, 10.85]
Info about DLP: 174.00[295.50, 113.50]
Info about TS-SNR: 19.29±5.20
Info about TS-CNR: 22.81[25.27, 18.40]
Info about ZF-SNR: 19.34±5.18
Info about ZF-CNR: 22.53[25.05, 19.30]


In [5]:
# 不同分组的患者基本信息,心率
p_info_76 = p_info[p_info['心率平均值']>75]
p_info_75 = p_info[p_info['心率平均值']<=75]

# Group 1
patient = p_info_76['影像号'].values
print('Number of patient = %.d'% (len(patient)))
print('Number of patients with stair-step artifacts = %.d'% (len(set(patient)&set(positive))))
print('Number of stair-step artifacts = %.d'% (data['患者编号'].isin(p_info_76['影像号']).sum()/4))
male1, female1 = p_info_76['性别'].value_counts()
print('Info about Gender: male = %.d (%.1f), female = %.d (%.1f)'% (male1,male1*100/len(patient),female1,female1*100/len(patient)))
# Group 2
patient = p_info_75['影像号'].values
print('Number of patient = %.d'% (len(patient)))
print('Number of patients with stair-step artifacts = %.d'% (len(set(patient)&set(positive))))
print('Number of stair-step artifacts = %.d'% (data['患者编号'].isin(p_info_75['影像号']).sum()/4))
male2, female2 = p_info_75['性别'].value_counts()
print('Info about Gender: male = %.d (%.1f), female = %.d (%.1f)'% (male2,male2*100/len(patient),female2,female2*100/len(patient)))

statistic,pvalue = stats.chi2_contingency(observed=[[male1,female1],[male2,female2]])[0:2]
print('Comparison for Gender: %.4f,%.4f'% (statistic,pvalue))

# Group 1 vs Group 2
for item in ['体重','身高（m)','BMI','造影剂总量','CTDI','SSDE','DLP']:
    buf1 = p_info_76[item].values
    buf2 = p_info_75[item].values
    # 正态性检验
    KS1, p_lili1 = lilliefors(buf1)
    KS2, p_lili2 = lilliefors(buf2)
    # Group 1
    if p_lili1 > 0.05:
        print('Info about '+item+': %.2f±%.2f'% (np.mean(buf1),np.std(buf1)))
    else:
        print('Info about '+item+': %.2f[%.2f, %.2f]'% 
              (np.median(buf1),np.percentile(buf1,[75,25])[0],np.percentile(buf1,[75,25])[1]))
    # Group 2
    if p_lili2 > 0.05:
        print('Info about '+item+': %.2f±%.2f'% (np.mean(buf2),np.std(buf2)))
    else:
        print('Info about '+item+': %.2f[%.2f, %.2f]'% 
              (np.median(buf2),np.percentile(buf2,[75,25])[0],np.percentile(buf2,[75,25])[1]))
    
    if p_lili1 > 0.05 and p_lili2 > 0.05:
    #方差齐性检验
        fVal, p_SD = stats.levene(buf1, buf2, center='mean')
        #独立样本t检验
        if p_SD > 0.05:
            t, p_final = stats.ttest_ind(buf1, buf2) 
        else:
            t, p_final = stats.ttest_ind(buf1, buf2, equal_var=False)
    else:
        #秩和检验
        t, p_final = stats.mannwhitneyu(buf1, buf2) 
    print('Comparison between groups for '+item+': %.4f' % (p_final))

Number of patient = 26
Number of patients with stair-step artifacts = 12
Number of stair-step artifacts = 27
Info about Gender: male = 15 (57.7), female = 11 (42.3)
Number of patient = 69
Number of patients with stair-step artifacts = 34
Number of stair-step artifacts = 85
Info about Gender: male = 37 (53.6), female = 32 (46.4)
Comparison for Gender: 0.0154,0.9012
Info about 体重: 70.69±12.23
Info about 体重: 70.86±11.47
Comparison between groups for 体重: 0.9524
Info about 身高（m): 1.66±0.08
Info about 身高（m): 1.66[1.72, 1.60]
Comparison between groups for 身高（m): 0.6061
Info about BMI: 25.49±2.99
Info about BMI: 25.59±3.12
Comparison between groups for BMI: 0.8870
Info about 造影剂总量: 65.00[70.00, 65.00]
Info about 造影剂总量: 65.00[70.00, 64.50]
Comparison between groups for 造影剂总量: 0.7023
Info about CTDI: 12.10[14.78, 9.46]
Info about CTDI: 12.40[22.60, 7.01]
Comparison between groups for CTDI: 0.9302
Info about SSDE: 16.80[20.90, 13.50]
Info about SSDE: 16.80[31.20, 9.81]
Comparison between groups f

In [6]:
# 不同分组的患者基本信息,BMI
p_info_26 = p_info[p_info['BMI']>25]
p_info_25 = p_info[p_info['BMI']<=25]

# Group 1
patient = p_info_26['影像号'].values
print('Number of patient = %.d'% (len(patient)))
print('Number of patients with stair-step artifacts = %.d'% (len(set(patient)&set(positive))))
print('Number of stair-step artifacts = %.d'% (data['患者编号'].isin(p_info_26['影像号']).sum()/4))
male1, female1 = p_info_26['性别'].value_counts()
print('Info about Gender: male = %.d (%.1f), female = %.d (%.1f)'% (male1,male1*100/len(patient),female1,female1*100/len(patient)))
# Group 2
patient = p_info_25['影像号'].values
print('Number of patient = %.d'% (len(patient)))
print('Number of patients with stair-step artifacts = %.d'% (len(set(patient)&set(positive))))
print('Number of stair-step artifacts = %.d'% (data['患者编号'].isin(p_info_25['影像号']).sum()/4))
male2, female2 = p_info_25['性别'].value_counts()
print('Info about Gender: male = %.d (%.1f), female = %.d (%.1f)'% (male2,male2*100/len(patient),female2,female2*100/len(patient)))

pvalue = stats.chi2_contingency(observed=[[male1,female1],[male2,female2]]).pvalue
print('Comparison for Gender: %.4f'% (pvalue))

# Group 1 vs Group 2
for item in ['体重','身高（m)','心率平均值','造影剂总量','CTDI','SSDE','DLP']:
    buf1 = p_info_26[item].values
    buf2 = p_info_25[item].values
    # 正态性检验
    KS1, p_lili1 = lilliefors(buf1)
    KS2, p_lili2 = lilliefors(buf2)
    # Group 1
    if p_lili1 > 0.05:
        print('Info about '+item+': %.2f±%.2f'% (np.mean(buf1),np.std(buf1)))
    else:
        print('Info about '+item+': %.2f[%.2f, %.2f]'% 
              (np.median(buf1),np.percentile(buf1,[75,25])[0],np.percentile(buf1,[75,25])[1]))
    # Group 2
    if p_lili2 > 0.05:
        print('Info about '+item+': %.2f±%.2f'% (np.mean(buf2),np.std(buf2)))
    else:
        print('Info about '+item+': %.2f[%.2f, %.2f]'% 
              (np.median(buf2),np.percentile(buf2,[75,25])[0],np.percentile(buf2,[75,25])[1]))
    
    if p_lili1 > 0.05 and p_lili2 > 0.05:
    #方差齐性检验
        fVal, p_SD = stats.levene(buf1, buf2, center='mean')
        #独立样本t检验
        if p_SD > 0.05:
            t, p_final = stats.ttest_ind(buf1, buf2) 
        else:
            t, p_final = stats.ttest_ind(buf1, buf2, equal_var=False)
    else:
        #秩和检验
        t, p_final = stats.mannwhitneyu(buf1, buf2) 
    print('Comparison between groups for '+item+': %.4f' % (p_final))

Number of patient = 52
Number of patients with stair-step artifacts = 22
Number of stair-step artifacts = 57
Info about Gender: male = 37 (71.2), female = 15 (28.8)
Number of patient = 43
Number of patients with stair-step artifacts = 24
Number of stair-step artifacts = 55
Info about Gender: male = 28 (65.1), female = 15 (34.9)
Comparison for Gender: 0.6830
Info about 体重: 77.00[82.00, 73.00]
Info about 体重: 60.00[67.00, 58.00]
Comparison between groups for 体重: 0.0000
Info about 身高（m): 1.70[1.73, 1.62]
Info about 身高（m): 1.65[1.70, 1.60]
Comparison between groups for 身高（m): 0.0882
Info about 心率平均值: 68.56±10.52
Info about 心率平均值: 68.84±10.74
Comparison between groups for 心率平均值: 0.8997
Info about 造影剂总量: 66.25[70.00, 65.00]
Info about 造影剂总量: 65.00[67.50, 62.50]
Comparison between groups for 造影剂总量: 0.0088
Info about CTDI: 15.30[25.32, 10.38]
Info about CTDI: 9.31[12.75, 6.04]
Comparison between groups for CTDI: 0.0002
Info about SSDE: 23.51±12.45
Info about SSDE: 13.90[17.80, 9.05]
Comparison 

# 医生结果一致性

In [7]:
for item in ['伪影等级评分','图像质量评分','医师诊断信心等级']:
    data1 = data[data['doctor']==1][item].values
    data2 = data[data['doctor']==2][item].values
    # 计算二者评分之间的Kappa系数
    kappa = cohen_kappa_score(data1,data2,weights='linear')
    
    np.random.seed(123)
    n_boot = 1000
    boot_kappas = []
    for _ in range(n_boot):
        idx = np.random.choice(len(data1), len(data1), replace=True)
        d1_boot = data1[idx]
        d2_boot = data2[idx]
        boot_kappa = cohen_kappa_score(d1_boot, d2_boot, weights='linear')
        boot_kappas.append(boot_kappa)
    
    ci_lower = np.percentile(boot_kappas, 2.5)
    ci_upper = np.percentile(boot_kappas, 97.5)
    
    print(f'{item} Kappa = {kappa:.4f}, 95% CI: [{ci_lower:.4f}, {ci_upper:.4f}]')

伪影等级评分 Kappa = 0.9627, 95% CI: [0.9419, 0.9811]
图像质量评分 Kappa = 0.9674, 95% CI: [0.9472, 0.9833]
医师诊断信心等级 Kappa = 0.9687, 95% CI: [0.9496, 0.9836]


# 主观参数 - barplot

In [8]:
# Doctor 1
new_data = data[data['doctor']==1]
save_name = 'comparison of doctor 1.pdf'
# 图像属性设置
sizeratio = 20
fig = plt.figure(figsize=(sizeratio*1.7,sizeratio))
plt.rcParams['font.sans-serif'] = ['Times New Roman']
plt.rcParams['axes.unicode_minus'] = False
colors = ['#CAF0F8','#90E0EF','#00B4D8','#0096C7','#0077B6']
width = 0.8
legendsize = sizeratio*2
textsize = sizeratio*1.8
ticksize = sizeratio*1.8
labelsize = sizeratio*2.5
psize = sizeratio*1.8

for i in range(len(name)):
    # 计算列联表做卡方检验，频率分布表后续做柱状图使用
    comp = pd.crosstab(new_data[name[i]],data['method'])
    print(comp)
    freq = comp.div(comp.sum()[0])
    
    plt.bar([i*3,i*3+1],freq.iloc[0:1].values[0],width=width,color=colors[0],label='1')
    plt.bar([i*3,i*3+1],freq.iloc[1:2].values[0],width=width,bottom=freq.iloc[0:1].sum(axis=0).values,color=colors[1],label='2')
    plt.bar([i*3,i*3+1],freq.iloc[2:3].values[0],width=width,bottom=freq.iloc[0:2].sum(axis=0).values,color=colors[2],label='3')
    plt.bar([i*3,i*3+1],freq.iloc[3:4].values[0],width=width,bottom=freq.iloc[0:3].sum(axis=0).values,color=colors[3],label='4')
    plt.bar([i*3,i*3+1],freq.iloc[4:5].values[0],width=width,bottom=freq.iloc[0:4].sum(axis=0).values,color=colors[4],label='5')
    if i == 0:
        plt.legend(frameon=False,markerscale=30,fontsize=legendsize)
    
    # 卡方检验，并将结果显示在图上
    statistic,p_value = stats.chi2_contingency(observed=comp)[0:2]
    print(name[i],statistic,p_value)
    if p_value < 0.001:
        plt.text(i*3+0.2,1.06,'p < 0.001',fontsize=psize)
    else:
        plt.text(i*3+0.2,1.06,'p = '+str(round(p_value,3)),fontsize=psize)
    
plt.tick_params(axis='x',length=0)
plt.ylabel('Frequency',fontsize=labelsize)
plt.ylim(0,1.15)
plt.xlim(-1,8.5)
plt.yticks(np.arange(0,1.2,0.2),[f'{i}%' for i in range(0,120,20)],fontsize=ticksize)

x_ticks = ['Standard','ZeeFree','Standard','ZeeFree','Standard','ZeeFree']
plt.xticks([0,1,3,4,6,7],x_ticks,rotation=20,fontsize=ticksize)
plt.text(-0.2,1.1,'Misalignment artifacts',fontsize=textsize)
plt.text(3,1.1,'Image quality',fontsize=textsize)
plt.text(5.7,1.1,'Diagnostic confidence',fontsize=textsize)

# 隐藏右和上边框
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
        
plt.savefig(save_path+save_name)
plt.close()

method  TS  ZF
伪影等级评分        
1       16   2
2       32   3
3       33   3
4       31  18
5        0  86
伪影等级评分 149.36643990929707 2.7828467066498035e-31
method  TS  ZF
图像质量评分        
1       28   1
2       44   9
3       33   6
4        7  21
5        0  75
图像质量评分 148.94344627396026 3.4286712791367843e-31
method    TS  ZF
医师诊断信心等级        
1         36   5
2         30   3
3         22   5
4         21  15
5          3  84
医师诊断信心等级 132.64743028830497 1.0571693596955431e-27


In [9]:
# 图像属性设置
sizeratio = 20
fig = plt.figure(figsize=(sizeratio*3.4,sizeratio*2))
plt.rcParams['font.sans-serif'] = ['Times New Roman']
plt.rcParams['axes.unicode_minus'] = False
width = 0.8
colors = ['#CAF0F8','#90E0EF','#00B4D8','#0096C7','#0077B6']
legendsize = sizeratio*2
textsize = sizeratio*1.8
ticksize = sizeratio*1.8
labelsize = sizeratio*2.5
psize = sizeratio*1.8
save_name = 'comparison of doctor 1 for subgroups.pdf'
# 图像数据
for index in range(4):
    new_data = data[data['doctor']==1]
    if index == 0:
        new_data = new_data[new_data['患者编号'].isin(p_info_76['影像号'])] # Doctor 1 with heartbeat>75
    elif index == 1:
        new_data = new_data[new_data['患者编号'].isin(p_info_75['影像号'])] # Doctor 1 with heartbeat<=75
    elif index == 2:
        new_data = new_data[new_data['患者编号'].isin(p_info_26['影像号'])] # Doctor 1 with BMI>25
    else:
        new_data = new_data[new_data['患者编号'].isin(p_info_25['影像号'])] # Doctor 1 with BMI>25
    
    plt.subplot(2,2,index+1)
    for i in range(len(name)):
        # 计算列联表做卡方检验，频率分布表后续做柱状图使用
        comp = pd.crosstab(new_data[name[i]],data['method'])
        print(comp)
        freq = comp.div(comp.sum()[0])

        plt.bar([i*3,i*3+1],freq.iloc[0:1].values[0],width=width,color=colors[0],label='1')
        plt.bar([i*3,i*3+1],freq.iloc[1:2].values[0],width=width,bottom=freq.iloc[0:1].sum(axis=0).values,color=colors[1],label='2')
        plt.bar([i*3,i*3+1],freq.iloc[2:3].values[0],width=width,bottom=freq.iloc[0:2].sum(axis=0).values,color=colors[2],label='3')
        plt.bar([i*3,i*3+1],freq.iloc[3:4].values[0],width=width,bottom=freq.iloc[0:3].sum(axis=0).values,color=colors[3],label='4')
        plt.bar([i*3,i*3+1],freq.iloc[4:5].values[0],width=width,bottom=freq.iloc[0:4].sum(axis=0).values,color=colors[4],label='5')
        if i == 0:
            plt.legend(frameon=False,markerscale=28,fontsize=legendsize)

        # 卡方检验，并将结果显示在图上
        statistic,p_value = stats.chi2_contingency(observed=comp)[0:2]
        print(name[i],statistic,p_value)
        if p_value < 0.001:
            plt.text(i*3+0.2,1.06,'p < 0.001',fontsize=psize)
        else:
            plt.text(i*3+0.2,1.06,'p = '+str(round(p_value,3)),fontsize=psize)

    plt.tick_params(axis='x',length=0)
    plt.ylabel('Frequency',fontsize=labelsize)
    plt.ylim(0,1.15)
    plt.xlim(-1,8.5)
    plt.yticks(np.arange(0,1.2,0.2),[f'{i}%' for i in range(0,120,20)],fontsize=ticksize)

    x_ticks = ['Standard','ZeeFree','Standard','ZeeFree','Standard','ZeeFree']
    plt.xticks([0,1,3,4,6,7],x_ticks,rotation=20,fontsize=ticksize)
    plt.text(-0.5,1.1,'Misalignment artifacts',fontsize=textsize)
    plt.text(3.1,1.1,'Image quality',fontsize=textsize)
    plt.text(5.6,1.1,'Diagnostic confidence',fontsize=textsize)

    # 隐藏右和上边框
    ax = plt.gca()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
        
plt.savefig(save_path+save_name)
plt.close()

method  TS  ZF
伪影等级评分        
1        3   0
2        8   0
3        7   0
4        9   7
5        0  20
伪影等级评分 38.25 9.950706635151513e-08
method  TS  ZF
图像质量评分        
1       11   0
2        7   1
3        6   2
4        3   7
5        0  17
图像质量评分 36.1 2.759812527300759e-07
method    TS  ZF
医师诊断信心等级        
1         10   0
2          6   0
3          4   2
4          7   6
5          0  19
医师诊断信心等级 35.743589743589745 3.267317303650489e-07
method  TS  ZF
伪影等级评分        
1       13   2
2       24   3
3       26   3
4       22  11
5        0  66
伪影等级评分 112.30804597701149 2.3424184741961534e-23
method  TS  ZF
图像质量评分        
1       17   1
2       37   8
3       27   4
4        4  14
5        0  58
图像质量评分 113.53118279569892 1.2843580671554977e-23
method    TS  ZF
医师诊断信心等级        
1         26   5
2         24   3
3         18   3
4         14   9
5          3  65
医师诊断信心等级 98.88979378567696 1.695006468329332e-20
method  TS  ZF
伪影等级评分        
1       10   0
2       16   1
3       17   2
4

In [10]:
# Doctor 2
new_data = data[data['doctor']==2]
save_name = 'comparison of doctor 2.pdf'
# 图像属性设置
sizeratio = 20
fig = plt.figure(figsize=(sizeratio*1.7,sizeratio))
plt.rcParams['font.sans-serif'] = ['Times New Roman']
plt.rcParams['axes.unicode_minus'] = False
colors = ['#CAF0F8','#90E0EF','#00B4D8','#0096C7','#0077B6']
width = 0.8
legendsize = sizeratio*2
textsize = sizeratio*1.8
ticksize = sizeratio*1.8
labelsize = sizeratio*2.5
psize = sizeratio*1.8

for i in range(len(name)):
    # 计算列联表做卡方检验，频率分布表后续做柱状图使用
    comp = pd.crosstab(new_data[name[i]],data['method'])
    print(comp)
    freq = comp.div(comp.sum()[0])
    
    plt.bar([i*3,i*3+1],freq.iloc[0:1].values[0],width=width,color=colors[0],label='1')
    plt.bar([i*3,i*3+1],freq.iloc[1:2].values[0],width=width,bottom=freq.iloc[0:1].sum(axis=0).values,color=colors[1],label='2')
    plt.bar([i*3,i*3+1],freq.iloc[2:3].values[0],width=width,bottom=freq.iloc[0:2].sum(axis=0).values,color=colors[2],label='3')
    plt.bar([i*3,i*3+1],freq.iloc[3:4].values[0],width=width,bottom=freq.iloc[0:3].sum(axis=0).values,color=colors[3],label='4')
    plt.bar([i*3,i*3+1],freq.iloc[4:5].values[0],width=width,bottom=freq.iloc[0:4].sum(axis=0).values,color=colors[4],label='5')
    if i == 0:
        plt.legend(frameon=False,markerscale=30,fontsize=legendsize)
    
    # 卡方检验，并将结果显示在图上
    p_value = stats.chi2_contingency(observed=comp).pvalue
#     if p_value < 0.001:
#         plt.text(i*3+0.2,1.06,'p < .001',fontsize=psize)
#     elif p_value <= 0.049 and p_value >= 0.045:
#         plt.text(i*3+0.2,1.06,'p = '+str(round(p_value,3))[1:5],fontsize=psize)
#     elif p_value >= 0.99:
#         plt.text(i*3+0.2,1.06,'p > .99',fontsize=psize)
#     elif p_value >= 0.01:
#         plt.text(i*3+0.2,1.06,'p = '+str(round(p_value,2))[1:4],fontsize=psize)
#     elif p_value < 0.01:
#         plt.text(i*3+0.2,1.06,'p = '+str(round(p_value,3))[1:5],fontsize=psize)
    if p_value < 0.001:
        plt.text(i*3+0.2,1.06,'p < 0.001',fontsize=psize)
    else:
        plt.text(i*3+0.2,1.06,'p = '+str(round(p_value,3)),fontsize=psize)

plt.tick_params(axis='x',length=0)
plt.ylabel('Frequency',fontsize=labelsize)
plt.ylim(0,1.15)
plt.xlim(-1,8.5)
plt.yticks(np.arange(0,1.2,0.2),[f'{i}%' for i in range(0,120,20)],fontsize=ticksize)

x_ticks = ['Standard','ZeeFree','Standard','ZeeFree','Standard','ZeeFree']
plt.xticks([0,1,3,4,6,7],x_ticks,rotation=20,fontsize=ticksize)
plt.text(-0.2,1.1,'Misalignment artifacts',fontsize=textsize)
plt.text(3,1.1,'Image quality',fontsize=textsize)
plt.text(5.7,1.1,'Diagnostic confidence',fontsize=textsize)

# 隐藏右和上边框
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
        
plt.savefig(save_path+save_name)
plt.close()

method  TS  ZF
伪影等级评分        
1       11   1
2       37   4
3       34   3
4       30  21
5        0  83
method  TS  ZF
图像质量评分        
1       30   1
2       43   9
3       31   5
4        8  21
5        0  76
method    TS  ZF
医师诊断信心等级        
1         35   5
2         33   3
3         20   3
4         22  17
5          2  84


In [11]:
# 图像属性设置
sizeratio = 20
fig = plt.figure(figsize=(sizeratio*3.4,sizeratio*2))
plt.rcParams['font.sans-serif'] = ['Times New Roman']
plt.rcParams['axes.unicode_minus'] = False
width = 0.8
colors = ['#CAF0F8','#90E0EF','#00B4D8','#0096C7','#0077B6']
legendsize = sizeratio*2
textsize = sizeratio*1.8
ticksize = sizeratio*1.8
labelsize = sizeratio*2.5
psize = sizeratio*1.8
save_name = 'comparison of doctor 2 for subgroups.pdf'
# 图像数据
for index in range(4):
    new_data = data[data['doctor']==2]
    if index == 0:
        new_data = new_data[new_data['患者编号'].isin(p_info_76['影像号'])] # Doctor 2 with heartbeat>75
    elif index == 1:
        new_data = new_data[new_data['患者编号'].isin(p_info_75['影像号'])] # Doctor 2 with heartbeat<=75
    elif index == 2:
        new_data = new_data[new_data['患者编号'].isin(p_info_26['影像号'])] # Doctor 2 with BMI>25
    else:
        new_data = new_data[new_data['患者编号'].isin(p_info_25['影像号'])] # Doctor 2 with BMI>25
    
    plt.subplot(2,2,index+1)
    for i in range(len(name)):
        # 计算列联表做卡方检验，频率分布表后续做柱状图使用
        comp = pd.crosstab(new_data[name[i]],data['method'])
        print(comp)
        freq = comp.div(comp.sum()[0])

        plt.bar([i*3,i*3+1],freq.iloc[0:1].values[0],width=width,color=colors[0],label='1')
        plt.bar([i*3,i*3+1],freq.iloc[1:2].values[0],width=width,bottom=freq.iloc[0:1].sum(axis=0).values,color=colors[1],label='2')
        plt.bar([i*3,i*3+1],freq.iloc[2:3].values[0],width=width,bottom=freq.iloc[0:2].sum(axis=0).values,color=colors[2],label='3')
        plt.bar([i*3,i*3+1],freq.iloc[3:4].values[0],width=width,bottom=freq.iloc[0:3].sum(axis=0).values,color=colors[3],label='4')
        plt.bar([i*3,i*3+1],freq.iloc[4:5].values[0],width=width,bottom=freq.iloc[0:4].sum(axis=0).values,color=colors[4],label='5')
        if i == 0:
            plt.legend(frameon=False,markerscale=28,fontsize=legendsize)

        # 卡方检验，并将结果显示在图上
        p_value = stats.chi2_contingency(observed=comp).pvalue
#     if p_value < 0.001:
#         plt.text(i*3+0.2,1.06,'p < .001',fontsize=psize)
#     elif p_value <= 0.049 and p_value >= 0.045:
#         plt.text(i*3+0.2,1.06,'p = '+str(round(p_value,3))[1:5],fontsize=psize)
#     elif p_value >= 0.99:
#         plt.text(i*3+0.2,1.06,'p > .99',fontsize=psize)
#     elif p_value >= 0.01:
#         plt.text(i*3+0.2,1.06,'p = '+str(round(p_value,2))[1:4],fontsize=psize)
#     elif p_value < 0.01:
#         plt.text(i*3+0.2,1.06,'p = '+str(round(p_value,3))[1:5],fontsize=psize)
        if p_value < 0.001:
            plt.text(i*3+0.2,1.06,'p < 0.001',fontsize=psize)
        else:
            plt.text(i*3+0.2,1.06,'p = '+str(round(p_value,3)),fontsize=psize)
    plt.tick_params(axis='x',length=0)
    plt.ylabel('Frequency',fontsize=labelsize)
    plt.ylim(0,1.15)
    plt.xlim(-1,8.5)
    plt.yticks(np.arange(0,1.2,0.2),[f'{i}%' for i in range(0,120,20)],fontsize=ticksize)

    x_ticks = ['Standard','ZeeFree','Standard','ZeeFree','Standard','ZeeFree']
    plt.xticks([0,1,3,4,6,7],x_ticks,rotation=20,fontsize=ticksize)
    plt.text(-0.5,1.1,'Misalignment artifacts',fontsize=textsize)
    plt.text(3.1,1.1,'Image quality',fontsize=textsize)
    plt.text(5.6,1.1,'Diagnostic confidence',fontsize=textsize)

    # 隐藏右和上边框
    ax = plt.gca()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
        
plt.savefig(save_path+save_name)
plt.close()

method  TS  ZF
伪影等级评分        
1        3   0
2        8   0
3        7   0
4        9   7
5        0  20
method  TS  ZF
图像质量评分        
1       11   0
2        7   1
3        6   2
4        3   7
5        0  17
method    TS  ZF
医师诊断信心等级        
1         10   0
2          6   0
3          4   2
4          7   6
5          0  19
method  TS  ZF
伪影等级评分        
1        8   1
2       29   4
3       27   3
4       21  14
5        0  63
method  TS  ZF
图像质量评分        
1       19   1
2       36   8
3       25   3
4        5  14
5        0  59
method    TS  ZF
医师诊断信心等级        
1         25   5
2         27   3
3         16   1
4         15  11
5          2  65
method  TS  ZF
伪影等级评分        
1        7   0
2       19   1
3       17   2
4       14  10
5        0  44
method  TS  ZF
图像质量评分        
1       19   0
2       23   5
3       12   2
4        3   7
5        0  43
method    TS  ZF
医师诊断信心等级        
1         17   1
2         22   2
3          9   3
4          9   4
5          0  47
method  TS  Z

# 客观参数 - boxplot

In [12]:
# 整体比较
SNR_TS = p_info['TS-SNR'].values
SNR_ZF = p_info['ZF-SNR'].values
CNR_TS = p_info['TS-CNR'].values
CNR_ZF = p_info['ZF-CNR'].values
save_name = 'comparison of objective results.pdf'
# 图像属性设置
sizeratio = 20
fig = plt.figure(figsize=(sizeratio*2,sizeratio))
plt.rcParams['font.sans-serif'] = ['Times New Roman']
plt.rcParams['axes.unicode_minus'] = False
colors = ['#00B4D8','#0077B6']
width = 0.5
legendsize = sizeratio*2
textsize = sizeratio*1.8
ticksize = sizeratio*1.8
labelsize = sizeratio*2.5
psize = sizeratio*1.8

plt.subplot(1,2,1)
# SNR-TS
plt.boxplot(SNR_TS,positions=[0],widths=width,meanline=True,showfliers=True,notch=True,patch_artist=True,
            boxprops=dict(facecolor=colors[0],color=colors[0]),capprops=dict(color=colors[0],lw=2),
            whiskerprops=dict(color=colors[0],lw=2),
            flierprops=dict(color=colors[0], markeredgecolor=colors[0],markerfacecolor=colors[0]),
            medianprops=dict(color=colors[0]))
# SNR-ZF
plt.boxplot(SNR_ZF,positions=[1],widths=width,meanline=True,showfliers=True,notch=True,patch_artist=True,
            boxprops=dict(facecolor=colors[1],color=colors[1]),capprops=dict(color=colors[1],lw=2),
            whiskerprops=dict(color=colors[1],lw=2),
            flierprops=dict(color=colors[1], markeredgecolor=colors[1],markerfacecolor=colors[1]),
            medianprops=dict(color=colors[1]))
# 卡方检验，并将结果显示在图上
statistic,p_value = testtest(SNR_TS,SNR_ZF)
print('SNR',statistic,p_value)
# if p_value < 0.001:
#     plt.text(0.3,np.max([np.max(SNR_TS),np.max(SNR_ZF)]),'p < .001',fontsize=psize)
# elif p_value <= 0.049 and p_value >= 0.045:
#     plt.text(0.3,np.max([np.max(SNR_TS),np.max(SNR_ZF)]),'p = '+str(round(p_value,3))[1:5],fontsize=psize)
# elif p_value >= 0.99:
#     plt.text(0.3,np.max([np.max(SNR_TS),np.max(SNR_ZF)]),'p > .99',fontsize=psize)
# elif p_value >= 0.01:
#     plt.text(0.3,np.max([np.max(SNR_TS),np.max(SNR_ZF)]),'p = '+str(round(p_value,2))[1:4],fontsize=psize)
# elif p_value < 0.01:
#     plt.text(0.3,np.max([np.max(SNR_TS),np.max(SNR_ZF)]),'p = '+str(round(p_value,3))[1:5],fontsize=psize)
if p_value < 0.001:
    plt.text(0.3,np.max([np.max(SNR_TS),np.max(SNR_ZF)]),'p < 0.001',fontsize=psize)
else:
    plt.text(0.3,np.max([np.max(SNR_TS),np.max(SNR_ZF)]),'p = '+str(round(p_value,3)),fontsize=psize)
    
plt.tick_params(axis='x',length=0)
plt.xlim(-1,2)
plt.yticks(fontsize=ticksize)
x_ticks = ['Standard','ZeeFree']
plt.xticks([0,1],x_ticks,rotation=20,fontsize=ticksize)
plt.ylabel('SNR',fontsize=labelsize)

# 隐藏右和上边框
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.subplot(1,2,2)
# CNR-TS
plt.boxplot(CNR_TS,positions=[0],widths=width,meanline=True,showfliers=True,notch=True,patch_artist=True,
            boxprops=dict(facecolor=colors[0],color=colors[0]),capprops=dict(color=colors[0],lw=2),
            whiskerprops=dict(color=colors[0],lw=2),
            flierprops=dict(color=colors[0], markeredgecolor=colors[0],markerfacecolor=colors[0]),
            medianprops=dict(color=colors[0]))
# CNR-ZF
plt.boxplot(CNR_ZF,positions=[1],widths=width,meanline=True,showfliers=True,notch=True,patch_artist=True,
            boxprops=dict(facecolor=colors[1],color=colors[1]),capprops=dict(color=colors[1],lw=2),
            whiskerprops=dict(color=colors[1],lw=2),
            flierprops=dict(color=colors[1], markeredgecolor=colors[1],markerfacecolor=colors[1]),
            medianprops=dict(color=colors[1]))
# 卡方检验，并将结果显示在图上
statistic,p_value = testtest(CNR_TS,CNR_ZF)
print('CNR',statistic,p_value)
# if p_value < 0.001:
#     plt.text(0.3,np.max([np.max(CNR_TS),np.max(CNR_ZF)]),'p < .001',fontsize=psize)
# elif p_value <= 0.049 and p_value >= 0.045:
#     plt.text(0.3,np.max([np.max(CNR_TS),np.max(CNR_ZF)]),'p = '+str(round(p_value,3))[1:5],fontsize=psize)
# elif p_value >= 0.99:
#     plt.text(0.3,np.max([np.max(CNR_TS),np.max(CNR_ZF)]),'p > .99',fontsize=psize)
# elif p_value >= 0.01:
#     plt.text(0.3,np.max([np.max(CNR_TS),np.max(CNR_ZF)]),'p = '+str(round(p_value,2))[1:4],fontsize=psize)
# elif p_value < 0.01:
#     plt.text(0.3,np.max([np.max(CNR_TS),np.max(CNR_ZF)]),'p = '+str(round(p_value,3))[1:5],fontsize=psize)
if p_value < 0.001:
    plt.text(0.3,np.max([np.max(CNR_TS),np.max(CNR_ZF)]),'p < 0.001',fontsize=psize)
else:
    plt.text(0.3,np.max([np.max(CNR_TS),np.max(CNR_ZF)]),'p = '+str(round(p_value,3)),fontsize=psize)
    
plt.tick_params(axis='x',length=0)
plt.xlim(-1,2)
plt.yticks(fontsize=ticksize)
x_ticks = ['Standard','ZeeFree']
plt.xticks([0,1],x_ticks,rotation=20,fontsize=ticksize)
plt.ylabel('CNR',fontsize=labelsize)

# 隐藏右和上边框
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
        
plt.savefig(save_path+save_name)
plt.close()

SNR -0.0635489664904824 0.949396885301171
CNR 4479.0 0.9306165518677987


# 错层伪影分布图

In [13]:
branch = ['RCA','LAD','LCX','LAD-D','RI','LCX-OM']
artifact = np.array([[44,8],[35,9],[24,4],[7,4],[1,1],[1,0]])

In [14]:
# 图像属性设置
sizeratio = 20
fig = plt.figure(figsize=(sizeratio,sizeratio*1.3))
plt.rcParams['font.sans-serif'] = ['Times New Roman']
plt.rcParams['axes.unicode_minus'] = False
width = 0.6
colors = ['#CAF0F8','#90E0EF','#00B4D8','#0096C7','#0077B6','blue']
legendsize = sizeratio*2
textsize = sizeratio*1.8
ticksize = sizeratio*1.8
labelsize = sizeratio*2.5
psize = sizeratio*1.8
save_name = 'Branches of stair-step artifacts.jpg'
# 图像数据
for i in range(len(branch)):
    if i == 0:
        plt.bar([0,1],artifact[i],width=width,color=colors[i],label=branch[i])
    else:
        plt.bar([0,1],artifact[i],width=width,bottom=artifact[0:i].sum(axis=0),color=colors[i],label=branch[i])

plt.tick_params(axis='x',length=0)
plt.ylabel('Artifacts number',fontsize=labelsize)
plt.xlim(-1,2)
plt.yticks(fontsize=ticksize)

x_ticks = ['Standard','ZeeFree']
plt.xticks([0,1],x_ticks,rotation=20,fontsize=ticksize)
plt.legend(frameon=False,markerscale=30,fontsize=legendsize)

# 隐藏右和上边框
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
        
plt.savefig(save_path+save_name, dpi=300)
plt.close()

# 补充ctFFR和斑块、脂肪相关数据

In [15]:
new_data = pd.read_excel(path+'Data/DataAnalysis-Sup-new.xlsx', sheet_name='Sheet1')
# ctFFR无法计算率比较
FFR_TS = (new_data['CT-FFR-TS']=='无法计算').sum()/new_data['CT-FFR-TS'].shape[0]
FFR_ZF = (new_data['CT-FFR-ZF']=='无法计算').sum()/new_data['CT-FFR-ZF'].shape[0]

print('无法计算率：TS组=%.2f%%, ZF组=%.2f%%' % (FFR_TS*100,FFR_ZF*100))

无法计算率：TS组=15.50%, ZF组=2.33%


In [16]:
# 每个feature比较ZF和TS的结果，需要进行配对检验
f_col = ['CT-FFR','斑块体积','斑块负荷','冠周脂肪CT值','冠周脂肪体积']

for i in range(len(f_col)):
    feature = f_col[i]
    _data = new_data[[feature+'-TS',feature+'-ZF']]
    _data = _data.loc[(_data[feature+'-TS']!='无法计算') & (_data[feature+'-ZF']!='无法计算')]

    data_1 = _data[feature+'-TS'].values.astype(float)
    data_2 = _data[feature+'-ZF'].values.astype(float)

    #正态性检验
    KS_1, p_lili_1 = lilliefors(data_1)
    if p_lili_1 > 0.05:
        print('Info about '+feature+'-TS: %.2f±%.2f'% (np.mean(data_1),np.std(data_1)))
    else:
        print('Info about '+feature+'-TS: %.2f[%.2f, %.2f]'% 
              (np.median(data_1),np.percentile(data_1,[75,25])[0],np.percentile(data_1,[75,25])[1]))
    KS_2, p_lili_2 = lilliefors(data_2)
    if p_lili_2 > 0.05:
        print('Info about '+feature+'-ZF: %.2f±%.2f'% (np.mean(data_2),np.std(data_2)))
    else:
        print('Info about '+feature+'-ZF: %.2f[%.2f, %.2f]'% 
              (np.median(data_2),np.percentile(data_2,[75,25])[0],np.percentile(data_2,[75,25])[1]))
    if p_lili_1 >0.05 and p_lili_2 >0.05:
        #配对t检验
        t, p_final = stats.ttest_rel(data_1, data_2) 
        if p_final<0.0001:
            print('配对设计检验结果of '+f_col[i]+' p value<0.0001, t值=%.4f' % t)
        else:
            print('配对设计检验结果of '+f_col[i]+' p value=%.4f, t值=%.4f' % (p_final,t))
    else:
        #秩和检验
        z, p_final = stats.wilcoxon(data_1, data_2)
        if p_final<0.0001:
            print('配对设计检验结果of '+f_col[i]+' p value<0.0001, z值=%.4f' % z)
        else:
            print('配对设计检验结果of '+f_col[i]+' p value=%.4f, z值=%.4f' % (p_final,z))

Info about CT-FFR-TS: 0.88[0.93, 0.80]
Info about CT-FFR-ZF: 0.91[0.95, 0.85]
配对设计检验结果of CT-FFR p value<0.0001, z值=530.5000
Info about 斑块体积-TS: 19.01[31.52, 10.52]
Info about 斑块体积-ZF: 11.80[25.35, 6.42]
配对设计检验结果of 斑块体积 p value<0.0001, z值=796.0000
Info about 斑块负荷-TS: 3.03[9.36, 0.00]
Info about 斑块负荷-ZF: 1.63[8.51, 0.00]
配对设计检验结果of 斑块负荷 p value=0.0008, z值=1064.5000
Info about 冠周脂肪CT值-TS: -85.57±9.19
Info about 冠周脂肪CT值-ZF: -85.04±9.26
配对设计检验结果of 冠周脂肪CT值 p value=0.0381, t值=-2.0952
Info about 冠周脂肪体积-TS: 1865.24±606.27
Info about 冠周脂肪体积-ZF: 1896.52±620.20
配对设计检验结果of 冠周脂肪体积 p value=0.0022, t值=-3.1194


In [17]:
# 配对箱型图
sizeratio = 20
plt.rcParams['font.sans-serif'] = ['Songti SC']
plt.rcParams['axes.unicode_minus'] = False
width = 0.6
colors = ['#CAF0F8','#90E0EF','#00B4D8','#0096C7','#0077B6','blue']
legendsize = sizeratio*2
textsize = sizeratio*1.8
ticksize = sizeratio*1.8
labelsize = sizeratio*2.5
psize = sizeratio*1.8
save_name = '配对箱型图.pdf'

ylabels = ['CT-FFR','斑块体积/cm3','斑块负荷','冠周脂肪CT值/HU','冠周脂肪体积/cm3']
fig = plt.figure(figsize=(sizeratio*5,sizeratio))
for i in range(len(f_col)):
    plt.subplot(1,len(f_col),i+1)
    
    feature = f_col[i]
    _data = new_data[[feature+'-TS',feature+'-ZF']]
    _data = _data.loc[(_data[feature+'-TS']!='无法计算') & (_data[feature+'-ZF']!='无法计算')]

    data_1 = _data[feature+'-TS'].values.astype(float)
    data_2 = _data[feature+'-ZF'].values.astype(float)
    plt.boxplot(data_1,positions=[0],widths=width,meanline=True,showfliers=True,notch=True,patch_artist=True,
            boxprops=dict(facecolor=colors[0],color=colors[0]),capprops=dict(color=colors[0],lw=2),
            whiskerprops=dict(color=colors[0],lw=2),
            flierprops=dict(color=colors[0], markeredgecolor=colors[0],markerfacecolor=colors[0]),
            medianprops=dict(color=colors[0]))
    plt.boxplot(data_2,positions=[1],widths=width,meanline=True,showfliers=True,notch=True,patch_artist=True,
            boxprops=dict(facecolor=colors[1],color=colors[1]),capprops=dict(color=colors[1],lw=2),
            whiskerprops=dict(color=colors[1],lw=2),
            flierprops=dict(color=colors[1], markeredgecolor=colors[1],markerfacecolor=colors[1]),
            medianprops=dict(color=colors[1]))
    
    plt.tick_params(axis='x',length=0)
    plt.xlim(-1,2)
    plt.yticks(fontsize=ticksize)
    x_ticks = ['Standard','ZeeFree']
    plt.xticks([0,1],x_ticks,rotation=20,fontsize=ticksize)
    plt.ylabel(ylabels[i],fontsize=labelsize)

    # 隐藏右和上边框
    ax = plt.gca()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.savefig(save_path+save_name)
plt.close()

# 补充-金标准测试结果

In [18]:
new_ICA = pd.read_excel(path+'Data/20260129-SR--ZF+ICA.xlsx', sheet_name='Sheet1')

new_ICA = new_ICA.replace('≥50',0)
new_ICA = new_ICA.replace('＜50',1)

print(sum(new_ICA['Standard'].values=='无法评估')/new_ICA['Standard'].values.shape[0])
print(sum(new_ICA['ZeeFree'].values=='无法评估')/new_ICA['ZeeFree'].values.shape[0])

0.5
0.020833333333333332


In [19]:
# 将无法评估作为第三个分类计算Kappa
new_ICA = new_ICA.replace('无法评估',2)
print(cohen_kappa_score(new_ICA['Standard'].astype(float),new_ICA['ICA'].astype(float)))
print(cohen_kappa_score(new_ICA['ZeeFree'].astype(float),new_ICA['ICA'].astype(float)))

0.05084745762711862
0.8476190476190476


# 检验效能

In [20]:
from statsmodels.stats.power import TTestIndPower

effect_size = 0.5  # 效应量 Cohen's d
alpha = 0.05       # 显著性水平
power = 0.8        # 检验效能

analysis = TTestIndPower()
n = analysis.solve_power(effect_size=effect_size, alpha=alpha, power=power, ratio=1)
print(f"每组样本量: {n:.1f}")

power = analysis.solve_power(effect_size=0.5, nobs1=95, alpha=0.05, ratio=1)
print(f"Power: {power:.10f}")

每组样本量: 63.8
Power: 0.9290010922


# 补充交互分析

In [21]:
import statsmodels.api as sm

df = data.copy()
df = df.replace(['TS','ZF'],[1,2])
df['HR'] = np.where(df['患者编号'].isin(p_info_76['影像号']), 2, 1)
df['BMI'] = np.where(df['患者编号'].isin(p_info_26['影像号']), 2, 1)
df['interaction1'] = df['method'] * df['doctor']
df['interaction2'] = df['method'] * df['HR']
df['interaction3'] = df['method'] * df['BMI']

# 不同医生对于评分的交互检验
print('Interactions for doctors')
for item in ['伪影等级评分','图像质量评分','医师诊断信心等级']:
    X = df[['method','doctor','interaction1']]
    X = sm.add_constant(X)
    y = df[item]

    model = sm.OLS(y, X).fit()

    print('Interaction for '+item+' = '+str(model.pvalues['interaction1']))
    
# 心率对于评分的交互检验
print('Interactions for subgroups of HRs')
for item in ['伪影等级评分','图像质量评分','医师诊断信心等级']:
    X = df[['method','doctor','interaction2']]
    X = sm.add_constant(X)
    y = df[item]

    model = sm.OLS(y, X).fit()

    print('Interaction for '+item+' = '+str(model.pvalues['interaction2']))
    
# BMI对于评分的交互检验
print('Interactions for subgroups of BMIs')
for item in ['伪影等级评分','图像质量评分','医师诊断信心等级']:
    X = df[['method','doctor','interaction3']]
    X = sm.add_constant(X)
    y = df[item]

    model = sm.OLS(y, X).fit()

    print('Interaction for '+item+' = '+str(model.pvalues['interaction3']))

Interactions for doctors
Interaction for 伪影等级评分 = 0.7541229016548453
Interaction for 图像质量评分 = 0.8393320851268126
Interaction for 医师诊断信心等级 = 0.8633462402361074
Interactions for subgroups of HRs
Interaction for 伪影等级评分 = 0.17768319595599968
Interaction for 图像质量评分 = 0.8850751852829571
Interaction for 医师诊断信心等级 = 0.5349586893740031
Interactions for subgroups of BMIs
Interaction for 伪影等级评分 = 0.4258783493625995
Interaction for 图像质量评分 = 0.9963455160631653
Interaction for 医师诊断信心等级 = 0.6930692863219418
